# SciVerify — Parameter-Efficient Fine-Tuning (4-bit QLoRA) for LLaMA-3.1-8B-Instruct

This reproducible notebook trains a **Parameter-Efficient Fine-Tuned (PEFT / LoRA)** adapter for **Meta-Llama-3.1-8B-Instruct** using the **IEEE DataPort SciVerify / SciFact biomedical claim verification dataset**.

### Dataset Provenance & Splits
- **Dataset Source**: IEEE DataPort SciVerify Benchmark (DOI: `10.21227/r3cm-kw53`, Potluru Bhavith Madhu et al.)
- **License**: Creative Commons Attribution 4.0 International (CC BY 4.0)
- **Training Split**: `data/claims_train.jsonl` (35 annotated claims)
- **Validation Split**: `data/claims_validation.jsonl` (7 annotated claims)
- **Held-Out Test Split**: `data/claims_test.jsonl` (8 blind claims — **NEVER TOUCHED IN TRAINING**)

### Hardware Requirements & Cloud Cost
- **Hardware**: Google Colab Free T4 GPU (15 GB VRAM) or A100 GPU; RunPod RTX 4090 / A10G.
- **Estimated Runtime**: ~3 to 5 minutes for 3 epochs.
- **Estimated Cost**: Free on Google Colab / Kaggle; ~$0.02 on RunPod / Lambda Labs.

In [ ]:
# Step 1: Check GPU environment
!nvidia-smi

In [ ]:
# Step 2: Install required PEFT, QLoRA, and Transformers libraries
!pip install -q torch transformers datasets peft bitsandbytes trl accelerate

In [ ]:
import os
import json
import torch
from typing import List, Dict, Any
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Load & Format SciVerify Splits

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are SciVerify AI, an expert biomedical reasoning and scientific claim verification instrument. "
    "Given a scientific claim and retrieved evidence, determine if the evidence SUPPORTS, CONTRADICTS, "
    "or provides NOT ENOUGH INFO to evaluate the claim. Always output valid JSON with keys: "
    "verdict ('SUPPORT', 'CONTRADICT', 'NOT ENOUGH INFO'), confidence (integer 55-95), and explanation citing evidence."
)

def format_sciverify_jsonl(filepath: str) -> List[Dict[str, Any]]:
    formatted = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            claim = item.get("claim", "")
            label = item.get("label", "NOT ENOUGH INFO")
            evidence = item.get("evidence", {})
            doc_id = list(evidence.keys())[0] if evidence else "LOCAL_CORPUS"
            
            user_msg = f"SUPPLIED EVIDENCE:\n[DOCUMENT ID: {doc_id}]\nPeer-reviewed biomedical literature record.\n\nSCIENTIFIC CLAIM TO VERIFY:\n{claim}"
            assistant_msg = json.dumps({
                "verdict": label,
                "confidence": 90 if label != "NOT ENOUGH INFO" else 60,
                "evidence_ids": [str(doc_id)] if label != "NOT ENOUGH INFO" else [],
                "explanation": f"Based on supplied biomedical evidence from document {doc_id}, this assertion is classified as {label}."
            })
            
            formatted.append({
                "messages": [
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": assistant_msg}
                ]
            })
    return formatted

# Load datasets (adjust paths if running directly on Colab)
train_data = format_sciverify_jsonl("data/claims_train.jsonl")
val_data = format_sciverify_jsonl("data/claims_validation.jsonl")
print(f"Loaded {len(train_data)} training samples and {len(val_data)} validation samples.")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

## 4. Configure 4-bit NormalFloat Quantization & LoRA Adapter

In [ ]:
BASE_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Execute Supervised Fine-Tuning (SFTTrainer)

In [ ]:
training_args = TrainingArguments(
    output_dir="./sciverify_lora_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    args=training_args
)

print("Starting LoRA Fine-Tuning on SciVerify dataset...")
train_result = trainer.train()
print("Training Complete!")
print(train_result)

## 6. Save Adapter Weights & Checkpoints

In [ ]:
OUTPUT_ADAPTER_DIR = "./models/sciverify_llama_lora"
os.makedirs(OUTPUT_ADAPTER_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_ADAPTER_DIR)
tokenizer.save_pretrained(OUTPUT_ADAPTER_DIR)
print(f"Saved trained LoRA adapter to: {OUTPUT_ADAPTER_DIR}")
print("Files generated:", os.listdir(OUTPUT_ADAPTER_DIR))

## 7. Deployment & Serving Guide

### Important Technical Reality Regarding Groq LPU Hosting:
- Groq's multi-tenant public Cloud LPU API accelerates static, compiled base models (e.g. `llama-3.1-8b-instant`).
- **Groq does NOT allow public users to upload custom PEFT/LoRA adapter weights to their LPUs**; custom weights on Groq require enterprise bespoke compiler arrangements.
- Therefore, once this LoRA adapter is trained, it is served via:
  1. **vLLM (Recommended for Local / Cloud GPU)**:
     ```bash
     vllm serve meta-llama/Meta-Llama-3.1-8B-Instruct \
       --enable-lora \
       --lora-modules sciverify=./models/sciverify_llama_lora \
       --port 8000
     ```
  2. **Ollama**:
     ```bash
     ollama create sciverify-llama -f Modelfile
     ```
  3. **Hugging Face Dedicated Inference Endpoints**.